# Limitless TCG Tournament Scraper
Scrapes team data from a Limitless tournament standings page.
No EVs on this site — pokemon vectors use the no-EV format.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pickle
import time
import os

In [ ]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
TOURNAMENT_URL = "https://play.limitlesstcg.com/tournament/69c30ae236f5b5c303dbce1c/standings"
OUTPUT_PATH = os.path.join(NOTEBOOK_DIR, "limitless_team_vectors.pkl")
HEADERS = {'User-Agent': 'Mozilla/5.0'}
BASE_URL = "https://play.limitlesstcg.com"

In [ ]:
# Collect all unique teamlist URLs from the standings page
resp = requests.get(TOURNAMENT_URL, timeout=15, headers=HEADERS)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, 'html.parser')

teamlist_urls = sorted(set(
    l['href'] for l in soup.find_all('a', href=True)
    if 'teamlist' in l['href'].lower()
))
print(f"Found {len(teamlist_urls)} unique teamlist URLs")

In [ ]:
def parse_teamlist_page(soup):
    """Parse a Limitless teamlist page into a list of pokemon vectors.
    
    Each vector: [Name, Ability, Item, Move1, Move2, Move3, Move4]
    (No EVs on this site)
    """
    pkmn_divs = soup.find_all('div', class_='pkmn')
    team = []

    for pkmn in pkmn_divs:
        # Name
        name_div = pkmn.find('div', class_='name')
        name_span = name_div.find('span') if name_div else None
        name = name_span.get_text(strip=True) if name_span else 'NONE'

        # Item
        item_div = pkmn.find('div', class_='item')
        item = item_div.get_text(strip=True) if item_div else 'NONE'

        # Ability
        ability_div = pkmn.find('div', class_='ability')
        ability = ability_div.get_text(strip=True) if ability_div else 'NONE'
        # Remove the "Ability: " prefix
        if ability.startswith('Ability:'):
            ability = ability[len('Ability:'):].strip()

        # Moves
        attacks_ul = pkmn.find('ul', class_='attacks')
        moves = []
        if attacks_ul:
            for li in attacks_ul.find_all('li'):
                moves.append(li.get_text(strip=True))
        while len(moves) < 4:
            moves.append('NONE')
        moves = moves[:4]

        team.append([name, ability, item, moves[0], moves[1], moves[2], moves[3]])

    return team

In [ ]:
all_teams = []
failed_urls = []

for i, url_path in enumerate(teamlist_urls):
    if (i + 1) % 200 == 0 or i == 0:
        print(f"Scraping {i + 1}/{len(teamlist_urls)}: {url_path.split('/')[-2]}", flush=True)
    try:
        resp = requests.get(f"{BASE_URL}{url_path}", timeout=15, headers=HEADERS)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        team = parse_teamlist_page(soup)
        if team:
            all_teams.append(team)
        else:
            failed_urls.append((url_path, 'empty parse'))
    except Exception as e:
        failed_urls.append((url_path, str(e)))
    time.sleep(0.3)

print(f"\nDone! Scraped {len(all_teams)} teams successfully.")
if failed_urls:
    print(f"Failed on {len(failed_urls)} URLs:")
    for url, reason in failed_urls[:10]:
        print(f"  {url} — {reason}")

In [ ]:
# Preview first team
if all_teams:
    print("Example team (first scraped):")
    for mon in all_teams[0]:
        print(f"  {mon}")

In [ ]:
with open(OUTPUT_PATH, 'wb') as f:
    pickle.dump(all_teams, f)

print(f"Saved {len(all_teams)} teams to {OUTPUT_PATH}")
if all_teams:
    print(f"Each pokemon vector has {len(all_teams[0][0])} entries")

# Verify reload
with open(OUTPUT_PATH, 'rb') as f:
    loaded = pickle.load(f)
print(f"Reloaded {len(loaded)} teams")